# DS-3002 <br> Data Mining - Assignment 2

**Muhammad Moiz Khalid** <br>
**23i-2552**<br>
**BDS-6C**

## Preprocessing

In [ ]:
import re, json, random, time, os
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.patches import Patch

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

os.makedirs("embeddings", exist_ok=True)
print("Imports OK | PyTorch:", torch.__version__)


## Data Loading & Tokenisation

In [ ]:
def load_articles(filepath):
    """Parse numbered [N] articles from cleaned.txt / raw.txt."""
    with open(filepath, encoding='utf-8') as f:
        content = f.read()
    parts = re.split(r'\[(\d+)\]', content)
    articles = {}
    for i in range(1, len(parts), 2):
        text = parts[i + 1].strip()
        if text:
            articles[int(parts[i])] = text
    return articles

def tokenize(text):
    """Keep only Urdu Unicode characters; split on whitespace."""
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)
    return [t for t in text.split() if t]

cleaned_articles = load_articles('cleaned.txt')
raw_articles = load_articles('raw.txt')
metadata = json.load(open('Metadata.json', encoding='utf-8'))

tokenized_cleaned = {aid: tokenize(t) for aid, t in cleaned_articles.items()}
tokenized_raw = {aid: tokenize(t) for aid, t in raw_articles.items()}

doc_ids = sorted(cleaned_articles.keys())

print(f"Cleaned articles : {len(cleaned_articles)}")
print(f"Raw articles : {len(raw_articles)}")
print(f"Metadata entries : {len(metadata)}")

### Topic Assignment (keyword-based, 5 categories)

In [ ]:
TOPIC_KEYWORDS = {
    'Politics':       ['الیکشن', 'حکومت', 'وزیر', 'پارلیمنٹ', 'سیاست',
                       'جماعت', 'ووٹ', 'وزیراعظم', 'صدر', 'اسمبلی'],
    'Sports':         ['کرکٹ', 'میچ', 'ٹیم', 'کھلاڑی', 'ورلڈکپ',
                       'فٹبال', 'کھیل', 'ٹورنامنٹ', 'وکٹ', 'گول'],
    'Economy':        ['مہنگائی', 'تجارت', 'بینک', 'بجٹ', 'معیشت',
                       'روپیہ', 'قرض', 'سرمایہ', 'ٹیکس', 'ڈالر'],
    'International':  ['اقوام', 'معاہدہ', 'امریکہ', 'چین', 'ایران',
                       'یوکرین', 'اسرائیل', 'روس', 'غیرملکی', 'تنازع'],
    'Health_Society': ['ہسپتال', 'بیماری', 'ویکسین', 'سیلاب', 'تعلیم',
                       'صحت', 'وبا', 'ڈاکٹر', 'آفت', 'امداد'],
}

def assign_topic(article_id):
    combined = (metadata.get(str(article_id), {}).get('title', '') +
                cleaned_articles.get(article_id, ''))
    scores = {t: sum(combined.count(k) for k in kws)
              for t, kws in TOPIC_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'International'

article_topics = {aid: assign_topic(aid) for aid in cleaned_articles}

print("Topic distribution:")
for t, c in sorted(Counter(article_topics.values()).items()):
    print(f"  {t:<20}: {c}")


## 1.1 TF-IDF Weighting

In [ ]:
MAX_VOCAB = 10_000

all_tokens = [tok for tokens in tokenized_cleaned.values() for tok in tokens]
token_freq = Counter(all_tokens)

top_vocab = [w for w, _ in token_freq.most_common(MAX_VOCAB)]
vocab     = {w: i for i, w in enumerate(['<UNK>'] + top_vocab)}
inv_vocab = {i: w for w, i in vocab.items()}

V = len(vocab)          # vocabulary size (incl. <UNK>)
D = len(cleaned_articles)
print(f"Vocabulary size (incl. <UNK>): {V}")
print(f"Number of documents          : {D}")


In [ ]:
# ── Term-Document matrix (raw counts) shape: V × D ──
print("Building term-document matrix ...")
word_doc_counts = np.zeros((V, D), dtype=np.float32)
for d_idx, aid in enumerate(doc_ids):
    for tok in tokenized_cleaned[aid]:
        word_doc_counts[vocab.get(tok, 0), d_idx] += 1

# TF: normalised by document length
doc_lengths = word_doc_counts.sum(axis=0, keepdims=True)
doc_lengths[doc_lengths == 0] = 1
TF = word_doc_counts / doc_lengths          # V × D

# DF & IDF
DF  = (word_doc_counts > 0).sum(axis=1)    # V
IDF = np.log(D / (1 + DF))                 # V  (standard formula from assignment)

# TF-IDF
tfidf_matrix = TF * IDF[:, np.newaxis]     # V × D

np.save('embeddings/tfidf_matrix.npy', tfidf_matrix)
print(f"TF-IDF matrix shape : {tfidf_matrix.shape}")
print("Saved → embeddings/tfidf_matrix.npy")


In [ ]:
print("Top-10 most discriminative words per topic (mean TF-IDF):\n")
for topic in TOPIC_KEYWORDS:
    topic_doc_indices = [i for i, aid in enumerate(doc_ids)
                         if article_topics[aid] == topic]
    if not topic_doc_indices:
        continue
    mean_tfidf = tfidf_matrix[:, topic_doc_indices].mean(axis=1)
    top10_ids  = np.argsort(mean_tfidf)[::-1][:15]
    top10_words = [inv_vocab[i] for i in top10_ids
                   if inv_vocab[i] != '<UNK>'][:10]
    print(f"  {topic}:")
    for rank, w in enumerate(top10_words, 1):
        print(f"    {rank:2d}. {w}")
    print()


## 1.2 Pointwise Mutual Information (PPMI)

In [ ]:
WINDOW_K     = 5
PPMI_VOCAB_N = 2000   # top-N words used in co-occurrence matrix

ppmi_vocab_words = top_vocab[:PPMI_VOCAB_N - 1]
ppmi_vocab  = {w: i + 1 for i, w in enumerate(ppmi_vocab_words)}
ppmi_vocab['<UNK>'] = 0
ppmi_V = len(ppmi_vocab)
inv_ppmi_vocab = {i: w for w, i in ppmi_vocab.items()}

print(f"PPMI vocab size : {ppmi_V}")
print(f"Context window  : k = {WINDOW_K}")
print("Building co-occurrence matrix ...")

cooc = np.zeros((ppmi_V, ppmi_V), dtype=np.float32)
for aid in doc_ids:
    ids = [ppmi_vocab.get(t, 0) for t in tokenized_cleaned[aid]]
    for i, center in enumerate(ids):
        if center == 0:
            continue
        left  = max(0, i - WINDOW_K)
        right = min(len(ids), i + WINDOW_K + 1)
        for j in range(left, right):
            if j != i and ids[j] != 0:
                cooc[center, ids[j]] += 1

# PPMI weighting
row_sums = cooc.sum(axis=1, keepdims=True)
col_sums = cooc.sum(axis=0, keepdims=True)
total    = cooc.sum()
row_sums[row_sums == 0] = 1
col_sums[col_sums == 0] = 1

with np.errstate(divide='ignore', invalid='ignore'):
    pmi = np.log2((cooc * total) / (row_sums * col_sums))
    pmi[~np.isfinite(pmi)] = 0

ppmi_matrix = np.maximum(pmi, 0)

np.save('embeddings/ppmi_matrix.npy', ppmi_matrix)
print(f"PPMI matrix shape : {ppmi_matrix.shape}")
print("Saved → embeddings/ppmi_matrix.npy")


In [ ]:
# ── t-SNE of 200 most frequent tokens ──
print("Running t-SNE (this may take ~1 min) ...")

tsne_words  = [w for w in ppmi_vocab_words if w in ppmi_vocab][:200]
tsne_ids    = [ppmi_vocab[w] for w in tsne_words]
vecs_200    = ppmi_matrix[tsne_ids, :]          # 200 × ppmi_V

# SVD to 50 dims first (reduces noise & speeds up t-SNE)
svd     = TruncatedSVD(n_components=50, random_state=42)
reduced = svd.fit_transform(vecs_200)

tsne   = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
coords = tsne.fit_transform(reduced)

# Per-word topic colour
word_topic_map = {}
for aid, tokens in tokenized_cleaned.items():
    tp = article_topics[aid]
    for tok in tokens:
        if tok not in word_topic_map:
            word_topic_map[tok] = Counter()
        word_topic_map[tok][tp] += 1

TOPIC_COLORS = {
    'Politics': 'red', 'Sports': 'blue', 'Economy': 'green',
    'International': 'orange', 'Health_Society': 'purple'
}

def word_color(w):
    if w not in word_topic_map:
        return 'gray'
    best = word_topic_map[w].most_common(1)[0][0]
    return TOPIC_COLORS.get(best, 'gray')

colors = [word_color(w) for w in tsne_words]

fig, ax = plt.subplots(figsize=(14, 10))
for i, (x, y) in enumerate(coords):
    ax.scatter(x, y, c=colors[i], s=18, alpha=0.7)
    if i % 10 == 0:
        ax.annotate(tsne_words[i], (x, y), fontsize=5, alpha=0.65)

legend_elements = [Patch(facecolor=v, label=k) for k, v in TOPIC_COLORS.items()] +                   [Patch(facecolor='gray', label='Other')]
ax.legend(handles=legend_elements, loc='upper right', title='Topic')
ax.set_title('t-SNE of 200 Most Frequent Tokens (PPMI vectors, colour-coded by topic)')
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
plt.tight_layout()
plt.savefig('embeddings/tsne_ppmi.png', dpi=150)
plt.show()
print("Saved → embeddings/tsne_ppmi.png")


In [ ]:
# ── Top-5 nearest neighbours by cosine similarity (PPMI space) ──
ppmi_norms  = np.linalg.norm(ppmi_matrix, axis=1, keepdims=True)
ppmi_norms[ppmi_norms == 0] = 1
ppmi_normed = ppmi_matrix / ppmi_norms

query_words_ppmi = ['پاکستان', 'کرکٹ', 'حکومت', 'معیشت', 'فوج',
                    'صحت', 'تعلیم', 'الیکشن', 'بینک', 'عدالت']

print("Top-5 nearest neighbours in PPMI space:\n")
for qw in query_words_ppmi:
    qid = ppmi_vocab.get(qw)
    if qid is None or qid == 0:
        print(f"  '{qw}': not in PPMI vocab")
        continue
    sims = ppmi_normed @ ppmi_normed[qid]
    top5 = [inv_ppmi_vocab[i] for i in np.argsort(sims)[::-1][1:6]]
    print(f"  '{qw}': {top5}")


## 2.1 Skip-gram Word2Vec

### Vocabulary & Noise Distribution

In [ ]:
# ── Noise distribution P_n(w) ∝ f(w)^(3/4) ──
freq_array = np.zeros(V, dtype=np.float32)
for w, cnt in token_freq.items():
    if w in vocab:
        freq_array[vocab[w]] = cnt
freq_array[0] = 0   # exclude <UNK> from noise

noise_dist = freq_array ** 0.75
noise_dist = noise_dist / noise_dist.sum()

print(f"Vocabulary size : {V}")
print(f"Noise table sum : {noise_dist.sum():.6f}  (should be 1.0)")
print("\nTop-10 most frequent words:")
for w, c in token_freq.most_common(10):
    print(f"  {w:<20} freq={c:6d}  p_noise={noise_dist[vocab.get(w,0)]:.6f}")


### Skip-gram Dataset

In [ ]:
# Hyperparameters
D_EMBED    = 100   # embedding dimension (C3)
W2V_WINDOW = 5     # context window size
K_NOISE    = 10    # negative samples per positive
BATCH_SIZE = 512
EPOCHS     = 5
LR         = 0.001

class SkipGramDataset(Dataset):
    """Generates (center, context, K_noise_negatives) triplets."""
    def __init__(self, articles, word2idx, window=5, K=10):
        self.pairs = []
        self.K     = K
        for tokens in articles.values():
            ids = [word2idx.get(t, 0) for t in tokens]
            for i, center in enumerate(ids):
                left  = max(0, i - window)
                right = min(len(ids), i + window + 1)
                for j in range(left, right):
                    if j != i:
                        self.pairs.append((center, ids[j]))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        center, context = self.pairs[idx]
        negatives = np.random.choice(len(noise_dist), size=self.K, p=noise_dist)
        return (torch.tensor(center,    dtype=torch.long),
                torch.tensor(context,   dtype=torch.long),
                torch.tensor(negatives, dtype=torch.long))

dataset_clean = SkipGramDataset(tokenized_cleaned, vocab, W2V_WINDOW, K_NOISE)
dataset_raw   = SkipGramDataset(tokenized_raw,     vocab, W2V_WINDOW, K_NOISE)

print(f"Skip-gram pairs (cleaned) : {len(dataset_clean):,}")
print(f"Skip-gram pairs (raw)     : {len(dataset_raw):,}")
